# Module 14 lab · RL post-training and reasoning

The code from every lesson of [this module](https://aicourses.dev/research/rl-and-reasoning), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**This lab needs a GPU.** Colab opens it on one; if not, choose Runtime → Change runtime type → T4 GPU.

**First time?** Read the [one-time setup](https://aicourses.dev/research/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 14.2 Learning from rewards by hand: a slot-machine game

You play a game where each choice pays out at random, keep track of what paid, and shift your choices towards the winners. A policy gradient is this habit written as a formula.

[Read the lesson](https://aicourses.dev/research/rl-and-reasoning/rewards-by-hand)

### Explore a little

The simplest player always pulls the machine with the best average so far, called **greedy**. A slightly braver one, **ε-greedy** (“epsilon-greedy”), does the same except that with a small chance ε, say 10%, it pulls a machine at random. Here is a simulation of both, plus a purely random player, each playing 1,000 pulls, 200 times over:

In [ ]:
import numpy as np
chances = np.array([0.3, 0.5, 0.7])   # hidden chance A, B, C pay out
rng = np.random.default_rng(0)

def play(eps, pulls=1000):
    wins, tries, total = np.zeros(3), np.zeros(3), 0
    for t in range(pulls):
        if t < 3:
            a = t                               # try each machine once
        elif rng.random() < eps:
            a = rng.integers(3)                 # explore
        else:
            a = np.argmax(wins / tries)     # exploit the best so far
        r = rng.random() < chances[a]
        wins[a] += r
        tries[a] += 1
        total += r
    return total

for eps in (0.0, 0.1, 1.0):
    totals = [play(eps) for _ in range(200)]
    print(eps, np.mean(totals), np.percentile(totals, 10))
# 0.0 593.945 299.0
# 0.1 663.36 627.6
# 1.0 499.88 480.0

### Shift your choices towards the winners

The arrow $\leftarrow$ means “is replaced by”. Worked example: all preferences start at 0, so each chance is $\frac{1}{3}$. You pull C and win 1 token while your average is 0.5, so $R - \bar{R} = 0.5$. With $\eta = 0.1$, C’s preference rises by $0.1 \times 0.5 \times \frac{2}{3} \approx 0.033$, and A’s and B’s fall by $0.1 \times 0.5 \times \frac{1}{3} \approx 0.017$. The new chances are about 0.328, 0.328 and 0.345: C is now slightly more likely.

In [ ]:
rng = np.random.default_rng(1)

def softmax(h):
    e = np.exp(h - h.max())
    return e / e.sum()

h = np.zeros(3)          # a preference score for each machine
avg, step = 0.0, 0.1     # running average payout; size of each nudge
for t in range(1, 1001):
    p = softmax(h)                       # preferences -> chances
    a = rng.choice(3, p=p)               # pull a machine at random
    r = float(rng.random() < chances[a])
    avg += (r - avg) / t                 # update the average payout
    h -= step * (r - avg) * p            # every machine, a little...
    h[a] += step * (r - avg)             # ...and the pulled one, more
    if t in (10, 100, 1000):
        print(t, softmax(h).round(3))
# 10 [0.315 0.324 0.361]
# 100 [0.158 0.215 0.627]
# 1000 [0.015 0.02  0.965]

## 14.3 Policy gradients from scratch: REINFORCE and baselines

A policy gradient makes attempts that scored well more likely, and ones that scored badly less likely. REINFORCE is the simplest form. Here a baseline is the average score, and it says what counts as scoring well. Most recent RL papers change only a small part of this method.

[Read the lesson](https://aicourses.dev/research/rl-and-reasoning/policy-gradients)

### The baseline

The log trick, used backwards, turns each $\pi(a)\, \nabla \log \pi(a)$ into $\nabla \pi(a)$, and the chances always add up to 1, which does not change. So a **baseline** changes nothing on average. What it changes is the noise. The reward minus the baseline is called the **advantage**: how much better than usual this attempt did. The check below compares single-pull gradient estimates for the three machines, starting from one third each, with and without a baseline, and then with every payout raised by a bonus of 10 tokens:

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
chances = np.array([0.3, 0.5, 0.7])
p = np.ones(3) / 3                   # the policy: one third each

def estimates(baseline, bonus=0.0, n=10_000):
    a = rng.choice(3, p=p, size=n)                 # n single pulls
    r = bonus + (rng.random(n) < chances[a])       # payout of each
    grad_logp = np.eye(3)[a] - p                   # slope of log pi(a)
    return (r - baseline)[:, None] * grad_logp

true = sum(p[a] * chances[a] * (np.eye(3)[a] - p) for a in range(3))
print(true.round(4))               # [-0.0667  0.      0.0667]
for bonus, b in ((0, 0), (0, 0.5), (10, 0), (10, 10.5)):
    g = estimates(b, bonus)
    print(bonus, b, g.mean(axis=0).round(3), g.std(axis=0).round(2))
# 0 0 [-0.068  0.001  0.066] [0.29 0.33 0.36]
# 0 0.5 [-0.062 -0.008  0.07 ] [0.23 0.24 0.22]
# 10 0 [-0.015 -0.083  0.099] [4.92 4.92 5.01]
# 10 10.5 [-0.065 -0.     0.065] [0.23 0.24 0.23]

### REINFORCE in PyTorch

In real code you do not write the slope by hand. You write a loss whose gradient is the policy gradient, and let autograd from Module 8 do the rest: the loss is minus the advantage times the log-probability of each sampled action. Here 16 pulls are drawn at each step, and the batch’s average payout is the baseline:

In [ ]:
import torch

torch.manual_seed(0)
chances = torch.tensor([0.3, 0.5, 0.7])
logits = torch.zeros(3, requires_grad=True)     # the policy's knobs
opt = torch.optim.SGD([logits], lr=0.5)
for step in range(301):
    logp = torch.log_softmax(logits, dim=0)
    a = torch.multinomial(logp.exp(), 16, replacement=True)  # 16 pulls
    r = (torch.rand(16) < chances[a]).float()   # their payouts
    baseline = r.mean()                         # this batch's average
    loss = -((r - baseline) * logp[a]).mean()   # REINFORCE
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 100 == 0:
        p = logp.exp().detach()
        entropy = -(p * p.log()).sum()
        print(step, p.numpy().round(3), round(entropy.item(), 3))
# 0 [0.333 0.333 0.333] 1.099
# 100 [0.025 0.039 0.936] 0.281
# 200 [0.012 0.022 0.966] 0.17
# 300 [0.009 0.017 0.974] 0.137

## 14.4 Reward models and RLHF

Reinforcement learning from human feedback (RLHF) first trains a reward model to guess which of two answers people prefer. Then it uses that score as the reward. You build one, as in the recipe behind ChatGPT, and see how pushing too hard on the score makes answers worse.

[Read the lesson](https://aicourses.dev/research/rl-and-reasoning/reward-models-and-rlhf)

### Build a tiny one

Below, each answer is described by two numbers: its quality and its length, both measured as distances from average. Simulated people prefer good answers, and like answers a little longer than average but not much longer: that is the `gold` score, the truth the reward model is trying to learn. A thousand pairs are labelled by these people, with the randomness of real raters, and a tiny reward model, a single linear layer, is trained on them:

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
spread = torch.tensor([0.5, 2.0])   # answers vary a little in quality,
                                    # a lot in length
def gold(x):                        # what people really prefer:
    q, L = x[..., 0], x[..., 1]     # quality, and length near average
    return q + 0.5 * L - 0.5 * L ** 2

a, b = torch.randn(1000, 2) * spread, torch.randn(1000, 2) * spread
a_wins = torch.rand(1000) < torch.sigmoid(gold(a) - gold(b))
chosen = torch.where(a_wins[:, None], a, b)
rejected = torch.where(a_wins[:, None], b, a)

rm = torch.nn.Linear(2, 1)                    # a tiny reward model
opt = torch.optim.Adam(rm.parameters(), lr=0.05)
for step in range(500):
    loss = -F.logsigmoid(rm(chosen) - rm(rejected)).mean()  # pairs
    opt.zero_grad()
    loss.backward()
    opt.step()
print(rm.weight.data)        # tensor([[0.4398, 0.1428]])
print((rm(chosen) > rm(rejected)).float().mean())   # tensor(0.6260)

### Pushing too hard

**Best-of-n** is the simplest way to optimise against a reward model: write $n$ answers, keep the one it scores highest. The more answers, the harder you push:

In [ ]:
for n in (1, 2, 4, 16, 64, 256):
    x = torch.randn(4000, n, 2) * spread      # n answers, 4,000 times
    with torch.no_grad():
        best = x[torch.arange(4000), rm(x).squeeze(-1).argmax(dim=1)]
        print(n, round(rm(best).mean().item(), 2),
              round(gold(best).mean().item(), 2))
# 1 -0.29 -2.11
# 2 -0.08 -1.39
# 4 0.09 -1.59
# 16 0.36 -3.13
# 64 0.57 -5.4
# 256 0.74 -7.88

## 14.5 PPO in plain terms: ratios, clipping and the KL leash

PPO improves a model in small, safe steps. It compares each answer’s new and old probability as a ratio, and clips that ratio so no single update goes too far. GRPO, in the next lesson, is built on it.

[Read the lesson](https://aicourses.dev/research/rl-and-reasoning/ppo)

### The clip

Worked example with $A = +1$ and $\epsilon = 0.2$: at $\rho = 1.1$ the objective is $\min(1.1, 1.1) = 1.1$; at $\rho = 1.3$ it is $\min(1.3, 1.2) = 1.2$, and stays 1.2 however far $\rho$ goes. The code below checks every case, using autograd to read the gradient each sample would give:

In [ ]:
import torch

def ppo_objective(ratio, adv, eps=0.2):
    clipped = torch.clamp(ratio, 1 - eps, 1 + eps)
    return torch.minimum(ratio * adv, clipped * adv)

for adv in (1.0, -1.0):
    for r in (0.7, 0.9, 1.0, 1.1, 1.3):
        ratio = torch.tensor(r, requires_grad=True)
        obj = ppo_objective(ratio, torch.tensor(adv))
        obj.backward()
        print(adv, r, round(obj.item(), 2), ratio.grad.item())

*The lesson shows:*

```text
1.0 0.7 0.7 1.0
1.0 0.9 0.9 1.0
1.0 1.0 1.0 1.0
1.0 1.1 1.1 1.0
1.0 1.3 1.2 0.0
-1.0 0.7 -0.8 0.0
-1.0 0.9 -0.9 -1.0
-1.0 1.0 -1.0 -1.0
-1.0 1.1 -1.1 -1.0
-1.0 1.3 -1.3 -1.0
```

### The KL leash

The clip limits each update. The leash from the last lesson limits the whole journey. InstructGPT gave each answer the reward model’s score minus $\beta$ times the log-ratio of the policy to the SFT model, summed over its tokens; averaged over sampled answers, that log-ratio is the KL divergence of lesson 9.5:

In [ ]:
score = 2.0                   # the reward model's score, at the end
logp_policy = torch.tensor([-1.2, -0.5, -2.0, -0.9])  # one answer
logp_ref = torch.tensor([-1.3, -0.9, -1.8, -0.9])     # SFT model
beta = 0.05
kl_per_token = logp_policy - logp_ref    # how far each token has moved
print(kl_per_token)   # tensor([ 0.1000,  0.4000, -0.2000,  0.0000])
reward = score - beta * kl_per_token.sum()
print(reward)         # tensor(1.9850)

## 14.6 GRPO and verifiable rewards

GRPO samples a group of answers to each question and scores each against the group’s average, so it needs no second network to judge them. With rewards from programs that check answers, such as a maths result, it is how open models learn to reason.

[Read the lesson](https://aicourses.dev/research/rl-and-reasoning/grpo)

### The group is the baseline

Worked example: four answers score 1, 0, 0 and 1. The mean is 0.5 and the standard deviation 0.5, so the advantages are $+1, -1, -1, +1$: the right answers are pushed up, the wrong ones down.

In [ ]:
import numpy as np

def group_advantages(rewards, eps=1e-4):
    r = np.asarray(rewards, dtype=float)
    return (r - r.mean()) / (r.std() + eps)   # eps: never divide by 0

for group in ([1, 0, 0, 1], [1, 0, 0, 0], [1, 1, 1, 1],
              [0.2, 0.9, 0.4, 0.5]):
    print(group, group_advantages(group).round(2))
# [1, 0, 0, 1] [ 1. -1. -1.  1.]
# [1, 0, 0, 0] [ 1.73 -0.58 -0.58 -0.58]
# [1, 1, 1, 1] [0. 0. 0. 0.]
# [0.2, 0.9, 0.4, 0.5] [-1.18  1.57 -0.39  0.  ]

### Rewards a program can check

In TRL, a reward function receives the batch’s completions, plus any other column of your dataset, and returns one number per completion:

In [ ]:
import re

def correctness(completions, answer, **kwargs):
    """1.0 if the number inside <answer>...</answer> is right."""
    rewards = []
    for text, gold in zip(completions, answer):
        found = re.findall(r"<answer>\s*(-?[\d,]+)\s*</answer>", text)
        ok = bool(found) and found[-1].replace(",", "") == gold
        rewards.append(1.0 if ok else 0.0)
    return rewards

samples = ["<think>12 x 5 = 60</think> <answer>60</answer>",
           "The answer is 60.",
           "<think>12 + 5</think> <answer>17</answer>",
           "<answer>1,060</answer>"]
print(correctness(samples, answer=["60", "60", "60", "1060"]))
# [1.0, 0.0, 0.0, 1.0]

### The full GRPO loss

Practice has moved on since. As of 2026, TRL’s `GRPOTrainer` defaults to 8 answers per question and a KL weight of 0, following later work that found the KL term not essential, which also saves loading the reference model. A 2025 paper, “Understanding R1-Zero-Like Training”, showed that the original way of averaging over each answer’s length makes wrong answers grow longer, and proposed a fix called Dr. GRPO; TRL offers it and other variants through `loss_type`. The core, a group baseline and a clipped ratio, is the same in all of them.

The lesson installs this first:

In [ ]:
!pip install trl peft

In [ ]:
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

args = GRPOConfig(output_dir="grpo-math", num_generations=8,
                  max_completion_length=512, learning_rate=1e-5,
                  fp16=True)                  # the T4 has no BF16
lora = LoraConfig(r=16, lora_alpha=32, target_modules="all-linear")
trainer = GRPOTrainer(model="Qwen/Qwen2.5-0.5B-Instruct",
                      reward_funcs=correctness, args=args,
                      train_dataset=dataset,  # "prompt" and "answer"
                      peft_config=lora)
trainer.train()

## 14.7 Reasoning models and test-time compute

Reasoning models write out long steps of thinking before they answer, and RL is what grows that habit. You also see how spending more computing at answer time, by sampling many answers and voting, raises scores, and how to measure that fairly.

[Read the lesson](https://aicourses.dev/research/rl-and-reasoning/reasoning-models)

### Measuring pass@k without fooling yourself

$\binom{n}{k}$, “n choose k”, is the number of ways to pick $k$ items from $n$. Worked example: $n = 10$, $c = 3$, $k = 5$. There are $\binom{7}{5} = 21$ ways to pick 5 from the 7 wrong answers and $\binom{10}{5} = 252$ ways to pick any 5, so pass@5 $= 1 - 21/252 \approx 0.917$.

In [ ]:
from math import comb

def pass_at_k(n, c, k):
    """Chance that k of n samples (c right) include a right one."""
    if n - c < k:
        return 1.0
    return 1 - comb(n - c, k) / comb(n, k)

print(pass_at_k(10, 3, 1), pass_at_k(10, 3, 5))
# 0.30000000000000004 0.9166666666666666

The simulation below makes 300 questions, each with its own chance of a right answer, and a wrong sample that is usually the same common mistake. It compares pass@k and maj@k from the same samples:

In [ ]:
import numpy as np
from collections import Counter

rng = np.random.default_rng(0)
Q, n = 300, 64                        # questions, samples per question
p_right = rng.beta(2, 2, size=Q)      # each question's chance of right
wrong = rng.choice([1, 2, 3], p=[0.6, 0.3, 0.1], size=(Q, n))
right = rng.random((Q, n)) < p_right[:, None]
answers = np.where(right, 0, wrong)   # answer 0 is the correct one

def majority_right(rows):
    return np.array([Counter(r).most_common(1)[0][0] == 0
                     for r in rows], dtype=float)

for k in (1, 4, 16, 64):
    pass_k = (answers[:, :k] == 0).any(axis=1).mean()
    maj_k = majority_right(answers[:, :k]).mean()
    print(k, round(pass_k, 3), round(maj_k, 3))
# 1 0.497 0.497
# 4 0.88 0.603
# 16 0.98 0.717
# 64 1.0 0.733

### Comparing fairly

- **Name the setting**: a score is pass@1, maj@64 or best-of-1,000, and they are not interchangeable. o1’s 74%, 83% and 93% are one model at three budgets.
- **Match the compute**: when comparing two methods, give them the same number of samples or tokens per question.
- **Put error bars on the gain**: majority-of-16 and a single answer are scored on the same questions, so use the paired interval of lesson 12.5. In the simulation:

In [ ]:
d = majority_right(answers[:, :16]) - (answers[:, 0] == 0)
se = d.std() / np.sqrt(Q)                 # paired, as in lesson 12.5
print(round(d.mean(), 3), round(d.mean() - 1.96 * se, 3),
      round(d.mean() + 1.96 * se, 3))     # 0.22 0.162 0.278

## 14.8 Reward hacking, and verifiers that resist it

Models find loopholes: ways to score well without doing the task. You find one yourself in a small game before the explanation, then learn to write checkers that are hard to fool, the core of building RL tasks.

[Read the lesson](https://aicourses.dev/research/rl-and-reasoning/reward-hacking)

### Watch a policy learn the hack

Below, a policy chooses between two strategies, honest solving, which succeeds 40% of the time, and spamming numbers, and is trained with the REINFORCE loss of lesson 14.3. Under the weak checker spam always pays; under a strong one it never does:

In [ ]:
import torch

def train(verifier_is_weak, steps=300):
    torch.manual_seed(0)
    logits = torch.zeros(2, requires_grad=True)  # 0: solve, 1: spam
    opt = torch.optim.SGD([logits], lr=0.5)
    for step in range(steps):
        logp = torch.log_softmax(logits, dim=0)
        a = torch.multinomial(logp.exp(), 16, replacement=True)
        solved = torch.rand(16) < 0.4            # solves 40% honestly
        spam_pays = torch.tensor(1.0 if verifier_is_weak else 0.0)
        r = torch.where(a == 0, solved.float(), spam_pays)
        loss = -((r - r.mean()) * logp[a]).mean()  # REINFORCE
        opt.zero_grad()
        loss.backward()
        opt.step()
    return logp.exp().detach().numpy().round(3)

print("weak verifier:", train(True))     # [0.006 0.994]
print("strong verifier:", train(False))  # [0.991 0.009]

### Writing checkers that resist

Here are the two checkers, tested against an honest answer and two cheats:

In [ ]:
import re

def weak(text, gold):          # right if the number appears anywhere
    return float(gold in re.findall(r"-?\d+", text))

def strong(text, gold):        # exactly one tagged answer, and right
    found = re.findall(r"<answer>\s*(-?\d+)\s*</answer>", text)
    return float(len(found) == 1 and found[0] == gold
                 and len(text) < 2000)

honest = "<think>12 x 7 = 84</think> <answer>84</answer>"
spam = " ".join(str(i) for i in range(1000))     # every number 0-999
hedge = "<answer>84</answer> or perhaps <answer>48</answer>"
cases = {"honest": honest, "spam": spam, "hedge": hedge}
for name, text in cases.items():
    print(name, weak(text, "84"), strong(text, "84"))
# honest 1.0 1.0
# spam 1.0 0.0
# hedge 1.0 0.0

## 14.9 Project: GRPO on math with a reward-hacking write-up

A real RL run you can explain, including the moment it learned to cheat.

[Read the lesson](https://aicourses.dev/research/rl-and-reasoning/project-grpo-math)

**This is the project.** These cells are starting points from the lesson; the steps between them are yours to write. A cell that uses a name from one of your steps stops with a NameError until you have written that step.

### Steps

1. **Set up.** In a Colab GPU runtime, `!pip install trl peft`, and vLLM too if your GPU has BF16 (lesson 14.6). Mount Google Drive and keep every run’s `output_dir` there. Load GSM8K’s training split for training and keep 500 test questions for evaluation, never trained on. Each GSM8K answer ends in `#### ` and the number; turn every row into a prompt and a clean final answer:

In [ ]:
SYSTEM = ("Think step by step inside <think> </think>, then give only "
          "the final number inside <answer> </answer>.")

def to_example(row):
    final = row["answer"].split("####")[-1].strip().replace(",", "")
    return {"prompt": [{"role": "system", "content": SYSTEM},
                       {"role": "user", "content": row["question"]}],
            "answer": final}

row = {"question": "A tray holds 12 buns. How many on 7 trays?",
       "answer": "12 * 7 = <<12*7=84>>84 buns.\n#### 84"}
print(to_example(row)["answer"])   # 84